# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.core.display_functions import clear_output

# Load data

In [ ]:
data_folder = "../../data/preprocessed-data"

df_eyewire2_roi_level = pd.read_hdf(os.path.join(data_folder, 'df_eyewire2_roi_level.h5'), key='dataframe')
df_eyewire2_field_level = pd.read_hdf(os.path.join(data_folder, 'df_eyewire2_field_level.h5'), key='dataframe')

In [ ]:
morph_folder = "../../data/morphological-data"
morph_spreadsheet_filename = "Eyewire II Proofreading - Calcium imaged cells - Proofreading 🧠.csv"

morph_df = pd.read_csv(os.path.join(os.path.join(morph_folder, morph_spreadsheet_filename)), header=3,
                        dtype={'Nuc ID': str})


In [ ]:
df = pd.concat([df_eyewire2_roi_level, morph_df], axis=1)
df['qfilt'] = (df['bar_qidx'] > 0.6) | (df['chirp_qidx'] > 0.45)

In [ ]:
print(list(df.columns))

# Plot data

In [ ]:
from eyewire2_functional_analysis.plot_dataframe import plot_df_chirp_and_bar

In [ ]:
fig_dir = './figures/responses_by_field'
os.makedirs(fig_dir, exist_ok=True)

for (field, cell_class), df_plot in df.groupby(['field', 'Cell Class']):
    clear_output()
    fig, axs = plot_df_chirp_and_bar(df=df_plot, title=f"{field}: All {cell_class}s", chirp_q_thresh=0.45, bar_q_thresh=0.6)
    plt.savefig(os.path.join(fig_dir, f'response_overview_{field}_{cell_class}.pdf'))
    plt.close(fig)

In [ ]:
fig_dir = './figures/responses_by_type'
os.makedirs(fig_dir, exist_ok=True)

def simplify_cell_type(cell_type_):
    # Return None if the cell type is None or NaN
    if cell_type_ is None or pd.isna(cell_type_):
        return None

    # Convert to string in case it's not already
    cell_type_ = str(cell_type_)

    # Remove question marks
    cell_type_ = cell_type_.replace('?', '')

    # Remove expressions of uncertainty
    uncertainty_phrases = ['maybe', 'I think', 'probably', 'or', '-ish']
    for phrase in uncertainty_phrases:
        # Split by phrase and take the first part
        if phrase in cell_type_.lower():
            parts = cell_type_.lower().split(phrase)
            cell_type_ = parts[0].strip()

    # If the cell type became empty after cleaning, return None
    if not cell_type_ or cell_type_.isspace():
        return None

    # Return the cleaned cell type
    return cell_type_.strip()

df['Cell Type simplified'] = df['Cell Type'].apply(simplify_cell_type)

for cell_type, df_plot in df.groupby('Cell Type simplified'):
    clear_output()
    fig, axs = plot_df_chirp_and_bar(df=df_plot.sort_values(['field', 'roi_id'], inplace=False),
                                     title=cell_type, chirp_q_thresh=0.45, bar_q_thresh=0.6)
    plt.savefig(os.path.join(fig_dir, f'response_overview_{cell_type}.pdf'))
    plt.close(fig)